# Dutch cycling accidents -- what the data actually shows

The Netherlands has the most extensive cycling network in the world, yet somewhere around 200 cyclists are killed on Dutch roads every year and tens of thousands more are injured. This notebook works through the data end-to-end: who crashes, where, under what conditions, and whether the infrastructure around them makes any measurable difference.

**Data sources joined in this dataset:**

| File | Rows | What it contains |
|------|------:|------------------|
| `accidents.parquet` | 56,743 | BRON microdata -- one row per cycling accident, 2022-2024 |
| `weather_daily.parquet` | 134,571 | KNMI daily observations at 38 stations, 2015-2024 |
| `weather_stations.csv` | 38 | Station coordinates and nearest-municipality mapping |
| `gemeenten.csv` | 342 | Municipality register -- population, area, CBS code |
| `gemeente_yearly_summary.csv` | 3,420 | Pre-joined summary per (municipality, year) |
| `cycling_infrastructure_osm.csv` | 342 | OSM cycling km and parking counts per municipality |

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as mtick
import seaborn as sns
import requests

PR = '#3D6E5C'; SE = '#7AAE99'; DK = '#1F3D33'; E1 = '#B0D4C5'; E2 = '#2A4D40'
SEQ = [PR, SE, DK, E1, E2]

sns.set_theme(style='white', font_scale=1.0)
mpl.rcParams.update({
    'figure.figsize': (11.0, 5.5), 'figure.dpi': 110, 'figure.facecolor': 'white',
    'savefig.dpi': 200, 'savefig.bbox': 'tight',
    'font.family': 'DejaVu Sans', 'font.size': 11,
    'axes.titlesize': 13, 'axes.titleweight': 'semibold', 'axes.titlecolor': DK,
    'axes.titlepad': 14, 'axes.labelsize': 11, 'axes.labelcolor': DK,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'xtick.color': DK, 'ytick.color': DK,
    'legend.fontsize': 10, 'legend.frameon': False,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.edgecolor': '#444444', 'axes.linewidth': 0.8,
    'axes.grid': True, 'grid.color': '#E8E8E8',
    'grid.linewidth': 0.6, 'grid.alpha': 0.8, 'axes.axisbelow': True,
    'lines.linewidth': 2.0, 'lines.markersize': 6,
    'axes.prop_cycle': mpl.cycler(color=SEQ),
})
sns.set_palette(SEQ)

# --- path detection ---
def _find_base():
    slug = 'dutch-cycling-accidents-infrastructure-weather'
    candidates = [
        f'/kaggle/input/{slug}',
        f'/kaggle/input/datasets/danushkumarv/{slug}',
    ]
    for p in candidates:
        if os.path.isfile(os.path.join(p, 'accidents.parquet')):
            return p
    # fallback: search /kaggle/input
    root = '/kaggle/input'
    if os.path.isdir(root):
        for d in sorted(os.listdir(root)):
            p = os.path.join(root, d)
            if os.path.isfile(os.path.join(p, 'accidents.parquet')):
                return p
        print('Contents of /kaggle/input:')
        for d in os.listdir(root):
            print(' ', d, '->', os.listdir(os.path.join(root, d))[:6])
    raise FileNotFoundError('accidents.parquet not found -- check dataset_sources in kernel-metadata.json')

BASE = _find_base()
print(f'Dataset found at: {BASE}')

In [ ]:
acc   = pd.read_parquet(f'{BASE}/accidents.parquet')
wx    = pd.read_parquet(f'{BASE}/weather_daily.parquet')
gm    = pd.read_csv(f'{BASE}/gemeenten.csv')
gy    = pd.read_csv(f'{BASE}/gemeente_yearly_summary.csv')
ws    = pd.read_csv(f'{BASE}/weather_stations.csv')
infra = pd.read_csv(f'{BASE}/cycling_infrastructure_osm.csv')

acc['year'] = acc['year'].astype(int)
wx['date']  = pd.to_datetime(wx['date'])

# translation maps
SEV_EN  = {'fatal': 'Fatal', 'injury': 'Injury', 'material damage only': 'Material damage'}
LIGHT_EN = {'Daglicht': 'Daylight', 'Duisternis': 'Darkness', 'Schemer': 'Twilight'}
ROAD_EN  = {
    'Rechte weg': 'Straight road', 'Kruispunt, 4 takken': '4-way intersection',
    'Kruispunt, 3 takken': 'T-junction', 'Rotonde': 'Roundabout',
    'Bocht': 'Curve', 'Rechte weg, gescheiden rijbanen': 'Divided highway',
}
ACC_EN = {
    'Flank': 'Side-impact', 'Frontaal': 'Head-on', 'Eenzijdig': 'Single-vehicle',
    'Kop/staart': 'Rear-end', 'Vast voorwerp': 'Fixed object',
    'Onbekend': 'Unknown', 'Geparkeerd voertuig': 'Parked vehicle',
    'Voetganger': 'Pedestrian', 'Los voorwerp': 'Loose object', 'Dier': 'Animal',
}
PARTY_EN = {
    'Personenauto': 'Car', 'Fiets': 'Bicycle', 'Bromfiets': 'Moped',
    'Bestelauto': 'Delivery van', 'e-bike': 'E-bike', 'Snorfiets': 'Speed pedelec',
    'Motor': 'Motorcycle', 'Bus': 'Bus', 'Vrachtauto': 'Truck',
    'Scootmobiel': 'Mobility scooter', 'Voetganger': 'Pedestrian',
    'Overig vast object': 'Fixed object', 'Los voorwerp': 'Loose object',
}
acc['sev_en']   = acc['severity_label'].map(SEV_EN).fillna(acc['severity_label'])
acc['light_en'] = acc['light_condition_label'].map(LIGHT_EN).fillna('Unknown')
acc['road_en']  = acc['road_situation'].map(ROAD_EN).fillna('Other')
acc['type_en']  = acc['accident_type'].map(ACC_EN).fillna('Other')
acc['p1_en']    = acc['party1_type'].map(PARTY_EN).fillna(acc['party1_type'])
acc['p2_en']    = acc['party2_type'].map(PARTY_EN).fillna(acc['party2_type'])

# yearly summary for accident years only
gy_acc = gy[gy['year'].isin([2022, 2023, 2024])].copy()
gy_acc['acc_per_100k']   = (gy_acc['accident_count']  / gy_acc['population'] * 100_000).round(2)
gy_acc['fatal_per_100k'] = (gy_acc['fatal_count']     / gy_acc['population'] * 100_000).round(4)

# municipality-level aggregates across 2022-2024
gm_agg = (
    gy_acc.groupby('gemeente_code', as_index=False)
    .agg(
        gemeente_name=('gemeente_name', 'first'),
        province=('province', 'first'),
        population=('population', 'first'),
        area_km2=('area_km2', 'first'),
        population_density_km2=('population_density_km2', 'first'),
        accident_count=('accident_count', 'sum'),
        fatal_count=('fatal_count', 'sum'),
        km_total=('km_total', 'first'),
        bicycle_parking_count=('bicycle_parking_count', 'first'),
    )
)
gm_agg['acc_per_100k']     = (gm_agg['accident_count'] / gm_agg['population'] * 100_000).round(2)
gm_agg['fatal_per_100k']   = (gm_agg['fatal_count']    / gm_agg['population'] * 100_000).round(4)
gm_agg['infra_km_per_km2'] = (gm_agg['km_total'] / gm_agg['area_km2']).round(3)

print(f'accidents      {len(acc):>7,} rows')
print(f'weather daily  {len(wx):>7,} rows')
print(f'gemeenten      {len(gm):>7,} rows')
print(f'yearly summary {len(gy):>7,} rows')
print(f'infra          {len(infra):>7,} rows')

---

## How bad do these accidents get?

BRON classifies every recorded accident into three severity levels. **Fatal** means someone died within 30 days. **Injury** means at least one person was hurt but survived. **Material damage only** means property damage with no casualties.

The 2022-2024 window is the current BRON WFS feed -- earlier years exist but require a separate API request.

In [ ]:
sev_counts = acc['sev_en'].value_counts()
yr_sev = acc.groupby(['year', 'sev_en']).size().unstack(fill_value=0)
yr_sev = yr_sev.reindex(columns=['Fatal', 'Injury', 'Material damage'], fill_value=0)
fatal_rate = yr_sev['Fatal'] / yr_sev.sum(axis=1) * 100

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

axes[0].pie(
    sev_counts.values, labels=sev_counts.index,
    colors=[PR, SE, E1], autopct='%1.1f%%', startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 10},
)
axes[0].set_title('Overall severity split (2022-2024)')

yr_sev.plot(kind='bar', ax=axes[1], color=[PR, SE, E1], width=0.6, edgecolor='white')
axes[1].set_title('Count by year')
axes[1].set_xlabel('')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[1].legend(loc='upper left', fontsize=9)

bars = axes[2].bar(fatal_rate.index.astype(str), fatal_rate.values, color=PR, edgecolor='white')
for bar, val in zip(bars, fatal_rate.values):
    axes[2].text(bar.get_x() + bar.get_width() / 2, val + 0.02,
                 f'{val:.2f}%', ha='center', fontsize=10, color=DK, fontweight='bold')
axes[2].set_title('Fatal share per year')
axes[2].set_ylim(0, fatal_rate.max() * 1.4)
axes[2].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.1f}%'))

plt.tight_layout()
plt.show()

total = len(acc)
for label, cnt in sev_counts.items():
    print(f'  {label:<22} {cnt:>6,}  ({cnt/total*100:5.1f}%)')

### Province breakdown

Zuid-Holland and Noord-Holland together account for roughly half the national accident total, which tracks with where most Dutch cycling happens. The more interesting question is the fatal _rate_ -- accidents per 100k population -- which tells a different story.

In [ ]:
PROV_ORDER = (
    acc.groupby('province')['severity_label']
    .apply(lambda s: (s == 'fatal').sum())
    .sort_values(ascending=True).index.tolist()
)
prov_sev = (
    acc.groupby(['province', 'sev_en']).size().unstack(fill_value=0)
    .reindex(columns=['Fatal', 'Injury', 'Material damage'], fill_value=0)
    .reindex(PROV_ORDER)
)
prov_pct = (prov_sev.div(prov_sev.sum(axis=1), axis=0) * 100).round(2)
prov_rate = (
    gm_agg.groupby('province')
    .apply(lambda d: d['accident_count'].sum() / d['population'].sum() * 100_000)
    .rename('acc_per_100k').sort_values(ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

prov_pct.plot(kind='barh', stacked=True, ax=axes[0],
              color=[PR, SE, E1], edgecolor='white')
axes[0].set_title('Severity mix by province (%)')
axes[0].set_xlabel('Share of accidents (%)')
axes[0].legend(['Fatal', 'Injury', 'Material damage'], loc='lower right', fontsize=9)
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))

bars = axes[1].barh(prov_rate.index, prov_rate.values, color=SE, edgecolor='white')
for bar, val in zip(bars, prov_rate.values):
    axes[1].text(val + 0.2, bar.get_y() + bar.get_height() / 2,
                 f'{val:.1f}', va='center', fontsize=9, color=DK)
axes[1].set_title('Accidents per 100k population (2022-2024 combined)')
axes[1].set_xlabel('Accidents per 100,000 residents')

plt.tight_layout()
plt.show()

---

## Where and when these accidents happen

Three things stand out consistently in cycling accident data worldwide: junctions are where most collisions occur, darkness multiplies severity, and speed limits are a near-perfect proxy for how bad the outcome will be.

In [ ]:
light_sev = (
    acc.groupby(['light_en', 'sev_en']).size().unstack(fill_value=0)
    .reindex(columns=['Fatal', 'Injury', 'Material damage'], fill_value=0)
)
light_pct = (light_sev.div(light_sev.sum(axis=1), axis=0) * 100).round(2)

speed_sev = (
    acc[acc['speed_limit_kmh'].notna()]
    .groupby(['speed_limit_kmh', 'sev_en']).size().unstack(fill_value=0)
    .reindex(columns=['Fatal', 'Injury', 'Material damage'], fill_value=0)
)
speed_total = speed_sev.sum(axis=1)
speed_pct   = (speed_sev.div(speed_total, axis=0) * 100).round(2)[speed_total >= 50]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

light_pct.plot(kind='bar', stacked=True, ax=axes[0],
               color=[PR, SE, E1], edgecolor='white', width=0.65)
for i, (lc, row) in enumerate(light_pct.iterrows()):
    axes[0].text(i, 2, f"{row['Fatal']:.2f}%",
                 ha='center', fontsize=9, color='white', fontweight='bold')
axes[0].set_title('Severity composition by light condition')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=0)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[0].legend(fontsize=9)

speed_pct.plot(kind='bar', stacked=True, ax=axes[1],
               color=[PR, SE, E1], edgecolor='white', width=0.65)
axes[1].set_title('Severity composition by speed limit')
axes[1].set_xlabel('Speed limit (km/h)')
axes[1].tick_params(axis='x', rotation=0)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Fatal rate by light condition:')
for lc, row in light_pct.iterrows():
    n = int(light_sev.loc[lc].sum())
    print(f'  {lc:<12}  {row["Fatal"]:.2f}% fatal  (n={n:,})')

### All 56,743 cycling accidents plotted across the Netherlands

Every point is a real accident. The Netherlands road network emerges from the density alone -- no basemap needed. Fatal accidents are rendered last, with a multi-layer glow so they don't disappear in the noise. The right panel switches to a hexbin density heatmap on the same data, which compresses the long tail of low-density areas and makes the Randstad hotspots sharper.

In [ ]:
from matplotlib.lines import Line2D

acc_geo = acc.dropna(subset=['latitude', 'longitude'])
acc_geo = acc_geo[
    (acc_geo['latitude'].between(50.5, 53.7)) &
    (acc_geo['longitude'].between(3.3, 7.3))
].copy()

fatal_pts    = acc_geo[acc_geo['severity_label'] == 'fatal']
injury_pts   = acc_geo[acc_geo['severity_label'] == 'injury']
material_pts = acc_geo[acc_geo['severity_label'] == 'material damage only']

BG = '#0D1117'
fig, axes = plt.subplots(1, 2, figsize=(20, 12), facecolor=BG)
for ax in axes:
    ax.set_facecolor(BG)
    ax.set_aspect('equal')
    ax.axis('off')

# === LEFT: pinpoint scatter -- every accident as a dot ===
axes[0].scatter(material_pts['longitude'], material_pts['latitude'],
                s=1.0, alpha=0.13, color='#2A5C4A', linewidths=0, rasterized=True)
axes[0].scatter(injury_pts['longitude'], injury_pts['latitude'],
                s=1.2, alpha=0.20, color='#C8963E', linewidths=0, rasterized=True)
# fatal: five concentric layers for glow
for sz, al in [(200, 0.018), (90, 0.055), (35, 0.17), (11, 0.55), (3, 1.0)]:
    axes[0].scatter(fatal_pts['longitude'], fatal_pts['latitude'],
                    s=sz, alpha=al, color='#FF1744', linewidths=0)

axes[0].set_title(
    f'All {len(acc_geo):,} cycling accidents\nNetherlands 2022-2024',
    color='white', fontsize=15, fontweight='bold', pad=18, loc='left')

legend_items = [
    Line2D([0],[0], marker='o', color='none', markerfacecolor='#2A5C4A',
           markersize=7, label=f'Material damage  ({len(material_pts):,})'),
    Line2D([0],[0], marker='o', color='none', markerfacecolor='#C8963E',
           markersize=7, label=f'Injury  ({len(injury_pts):,})'),
    Line2D([0],[0], marker='o', color='none', markerfacecolor='#FF1744',
           markersize=9, label=f'Fatal  ({len(fatal_pts)})'),
]
axes[0].legend(handles=legend_items, loc='lower left',
               frameon=False, labelcolor='#CCCCCC', fontsize=11, handletextpad=0.4)

axes[0].text(0.98, 0.02,
             f'Accident rate: {len(fatal_pts)/len(acc_geo)*100:.2f}% fatal',
             transform=axes[0].transAxes, color='#666666', fontsize=9,
             ha='right', va='bottom')

# === RIGHT: hexbin density + fatal glow in cyan ===
hb = axes[1].hexbin(
    acc_geo['longitude'], acc_geo['latitude'],
    gridsize=140, bins='log', cmap='inferno',
    mincnt=1, alpha=0.93, linewidths=0,
)
for sz, al in [(240, 0.012), (110, 0.045), (42, 0.15), (14, 0.65), (4, 1.0)]:
    axes[1].scatter(fatal_pts['longitude'], fatal_pts['latitude'],
                    s=sz, alpha=al, color='#00E5FF', linewidths=0)

axes[1].set_title(
    'Accident density -- log hexbin\nfatal accidents in cyan',
    color='white', fontsize=15, fontweight='bold', pad=18, loc='left')

cb = fig.colorbar(hb, ax=axes[1], shrink=0.30, pad=0.01,
                   aspect=18, location='right')
cb.set_label('log(accidents per cell)', color='#888888', fontsize=9)
cb.ax.yaxis.set_tick_params(color='#888888', labelcolor='#888888', labelsize=8)
cb.outline.set_edgecolor('#333333')

axes[1].text(0.98, 0.02, '140 x 140 hexagons',
             transform=axes[1].transAxes, color='#555555', fontsize=9,
             ha='right', va='bottom')

fig.text(0.5, 0.012,
         'Coordinates: WGS84  |  BRON microdata, Rijkswaterstaat  |  2022-2024',
         ha='center', color='#3A3A3A', fontsize=9)

plt.subplots_adjust(left=0.005, right=0.96, top=0.93, bottom=0.04, wspace=0.04)
plt.show()


### Junctions vs straight roads -- which is actually more dangerous?

In [ ]:
road_stats = (
    acc.groupby('road_en')
    .agg(total=('severity_label', 'count'),
         fatal=('severity_label', lambda s: (s == 'fatal').sum()))
    .assign(fatal_pct=lambda d: d['fatal'] / d['total'] * 100)
    .sort_values('total', ascending=False).head(7)
)
type_cnt = acc['type_en'].value_counts().head(8)

fig, axes = plt.subplots(1, 2, figsize=(17, 5.5))

ax2 = axes[0].twinx()
axes[0].bar(road_stats.index, road_stats['total'], color=E1, edgecolor='white', label='Total')
ax2.plot(road_stats.index, road_stats['fatal_pct'],
         marker='o', color=PR, linewidth=2.5, label='Fatal %')
ax2.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.1f}%'))
axes[0].set_title('Accident count and fatal rate by road situation')
axes[0].set_ylabel('Accident count')
ax2.set_ylabel('Fatal rate (%)', color=PR)
axes[0].tick_params(axis='x', rotation=25)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
l1, lbl1 = axes[0].get_legend_handles_labels()
l2, lbl2 = ax2.get_legend_handles_labels()
axes[0].legend(l1 + l2, lbl1 + lbl2, fontsize=9)

bars = axes[1].barh(type_cnt.index[::-1], type_cnt.values[::-1], color=SE, edgecolor='white')
for bar, val in zip(bars, type_cnt.values[::-1]):
    axes[1].text(val + 50, bar.get_y() + bar.get_height() / 2,
                 f'{val:,}', va='center', fontsize=9, color=DK)
axes[1].set_title('Accident type distribution')
axes[1].set_xlabel('Number of accidents')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()

---

## Who hits cyclists?

Every BRON record stores `party1_type` (the vehicle that was registered as the triggering party) and `party2_type` (the other vehicle). Because this dataset filters for cycling-involved accidents, at least one side is always a bicycle or e-bike.

In [ ]:
p1_top = acc['p1_en'].value_counts().head(8)
p2_top = acc['p2_en'].value_counts().head(8)

p1_fatal = (
    acc.groupby('p1_en')
    .agg(total=('severity_label', 'count'),
         fatal=('severity_label', lambda s: (s == 'fatal').sum()))
    .query('total >= 100')
    .assign(fatal_pct=lambda d: d['fatal'] / d['total'] * 100)
    .sort_values('fatal_pct', ascending=True)
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

bars1 = axes[0].barh(p1_top.index[::-1], p1_top.values[::-1], color=PR, edgecolor='white')
for bar, val in zip(bars1, p1_top.values[::-1]):
    axes[0].text(val + 80, bar.get_y() + bar.get_height() / 2,
                 f'{val:,}', va='center', fontsize=8, color=DK)
axes[0].set_title('Party 1 (triggering vehicle)')
axes[0].set_xlabel('Accidents')
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

bars2 = axes[1].barh(p2_top.index[::-1], p2_top.values[::-1], color=SE, edgecolor='white')
for bar, val in zip(bars2, p2_top.values[::-1]):
    axes[1].text(val + 80, bar.get_y() + bar.get_height() / 2,
                 f'{val:,}', va='center', fontsize=8, color=DK)
axes[1].set_title('Party 2 (other vehicle)')
axes[1].set_xlabel('Accidents')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

bars3 = axes[2].barh(p1_fatal.index, p1_fatal['fatal_pct'], color=E2, edgecolor='white')
for bar, val in zip(bars3, p1_fatal['fatal_pct'].values):
    axes[2].text(val + 0.05, bar.get_y() + bar.get_height() / 2,
                 f'{val:.1f}%', va='center', fontsize=8, color=DK)
axes[2].set_title('Fatal rate when party 1 is...')
axes[2].set_xlabel('Fatal share (%)')
axes[2].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.1f}%'))

plt.tight_layout()
plt.show()

### E-bike involvement is rising year over year

E-bikes now account for a growing slice of cycling accidents. This is partly a usage effect (more e-bikes on the road) and partly a speed effect (e-bikes travel faster, changing collision dynamics).

In [ ]:
yr_mode = acc.groupby('year').agg(
    total=('accident_id', 'count'),
    bicycle=('party1_type', lambda s: (s == 'Fiets').sum()),
    ebike_p1=('party1_type', lambda s: (s == 'e-bike').sum()),
    ebike_p2=('party2_type', lambda s: (s == 'e-bike').sum()),
    moped=('party1_type', lambda s: s.isin(['Bromfiets', 'Snorfiets']).sum()),
)
yr_mode['ebike'] = yr_mode['ebike_p1'] + yr_mode['ebike_p2']
yr_mode['ebike_share'] = yr_mode['ebike'] / yr_mode['total'] * 100

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for col, color, label in [
        ('bicycle', SE, 'Traditional bicycle'),
        ('ebike', PR, 'E-bike involved'),
        ('moped', E2, 'Moped / speed pedelec'),
]:
    axes[0].plot(yr_mode.index.astype(str), yr_mode[col],
                 marker='o', color=color, label=label)
axes[0].set_title('Vehicle involvement count by year')
axes[0].set_ylabel('Accidents')
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].legend(fontsize=9)

bars = axes[1].bar(yr_mode.index.astype(str), yr_mode['ebike_share'],
                    color=PR, edgecolor='white')
for bar, val in zip(bars, yr_mode['ebike_share']):
    axes[1].text(bar.get_x() + bar.get_width() / 2, val + 0.05,
                 f'{val:.1f}%', ha='center', fontsize=10, color=DK, fontweight='bold')
axes[1].set_title('E-bike share of all cycling accidents')
axes[1].set_ylabel('E-bike involvement (%)')
axes[1].set_ylim(0, yr_mode['ebike_share'].max() * 1.3)

plt.tight_layout()
plt.show()

for yr, row in yr_mode.iterrows():
    print(f'  {yr}: {int(row["ebike"]):>5,} e-bike accidents  ({row["ebike_share"]:.1f}% of total)')

### Alcohol

In [ ]:
alc = acc[acc['alcohol_indicated']].copy()
alc_rate = acc.groupby('sev_en')['alcohol_indicated'].mean() * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(alc['sev_en'].value_counts().index,
             alc['sev_en'].value_counts().values,
             color=[PR, SE, E1], edgecolor='white')
for i, val in enumerate(alc['sev_en'].value_counts().values):
    axes[0].text(i, val + 2, str(int(val)), ha='center', fontsize=10,
                 color=DK, fontweight='bold')
axes[0].set_title('Alcohol-indicated by severity')
axes[0].set_ylabel('Accident count')

axes[1].bar(alc_rate.index, alc_rate.values, color=[PR, SE, E1], edgecolor='white')
for i, val in enumerate(alc_rate.values):
    axes[1].text(i, val + 0.02, f'{val:.2f}%', ha='center', fontsize=10,
                 color=DK, fontweight='bold')
axes[1].set_title('Alcohol rate within each severity class')
axes[1].set_ylabel('Alcohol-indicated (%)')

plt.tight_layout()
plt.show()
print(f'Total alcohol-indicated: {len(alc):,}  ({len(alc)/len(acc)*100:.2f}% of all accidents)')

---

## Weather effects

BRON records the weather condition at the accident scene. The first thing to know: most accidents happen in clear weather -- not because clear weather is more dangerous, but because that is when most cycling happens. The fatality rate within each weather category is more informative.

In [ ]:
wx_sev = (
    acc.groupby(['weather_label', 'sev_en']).size().unstack(fill_value=0)
    .reindex(columns=['Fatal', 'Injury', 'Material damage'], fill_value=0)
)
wx_pct = (wx_sev.div(wx_sev.sum(axis=1), axis=0) * 100).round(2)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

wx_sev.plot(kind='bar', ax=axes[0], color=[PR, SE, E1], edgecolor='white', width=0.65)
axes[0].set_title('Raw count by weather condition')
axes[0].set_xlabel('')
axes[0].tick_params(axis='x', rotation=20)
axes[0].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].legend(fontsize=9)

wx_pct.plot(kind='bar', stacked=True, ax=axes[1], color=[PR, SE, E1], edgecolor='white', width=0.65)
for i, (cond, row) in enumerate(wx_pct.iterrows()):
    axes[1].text(i, 2, f"{row['Fatal']:.2f}%", ha='center', fontsize=9,
                 color='white', fontweight='bold')
axes[1].set_title('Severity composition by weather (%)')
axes[1].tick_params(axis='x', rotation=20)
axes[1].yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:.0f}%'))
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### National climate overview -- 10 years of KNMI data

The `weather_daily` file covers 38 stations from 2015 through 2024. Here we show the monthly temperature pattern across years and the annual precipitation variation.

In [ ]:
wx_yr = wx.copy()
wx_yr['year']  = wx_yr['date'].dt.year
wx_yr['month'] = wx_yr['date'].dt.month

monthly_temp = (
    wx_yr[wx_yr['year'] >= 2015]
    .groupby(['year', 'month'])['temp_mean_c'].mean().reset_index()
)
pivot = monthly_temp.pivot(index='year', columns='month', values='temp_mean_c')

ann_precip = (
    wx_yr[wx_yr['year'] >= 2015]
    .groupby('year')['precip_mm'].sum().div(38).reset_index(name='precip_mm')
)

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

cmap_temp = mcolors.LinearSegmentedColormap.from_list(
    'temp', ['#A8DAE3', '#E8E8E8', '#C75B3F'], N=256)
im = axes[0].imshow(pivot.values, aspect='auto', cmap=cmap_temp)
axes[0].set_xticks(range(12))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'], fontsize=9)
axes[0].set_yticks(range(len(pivot.index)))
axes[0].set_yticklabels(pivot.index.tolist(), fontsize=9)
axes[0].set_title('Mean daily temperature (deg C) by month and year')
plt.colorbar(im, ax=axes[0], shrink=0.8, label='Temp (deg C)')
axes[0].grid(False)

bars = axes[1].bar(ann_precip['year'].astype(str), ann_precip['precip_mm'],
                    color=SE, edgecolor='white')
axes[1].axhline(ann_precip['precip_mm'].mean(), color=PR, linewidth=1.5,
                 linestyle='--', label='2015-2024 mean')
axes[1].set_title('Mean annual precipitation per station (mm)')
axes[1].set_ylabel('Precipitation (mm)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

### Does more rain mean more accidents per municipality?

In [ ]:
gy_wx = gy_acc.dropna(subset=['weather_annual_precip_mm', 'weather_mean_temp_mean_c']).copy()
gy_wx = gy_wx[gy_wx['population'] > 0]
gy_wx['acc_per_100k'] = (gy_wx['accident_count'] / gy_wx['population'] * 100_000).round(2)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

for yr, grp in gy_wx.groupby('year'):
    axes[0].scatter(grp['weather_annual_precip_mm'], grp['acc_per_100k'],
                    alpha=0.7, s=55, label=str(yr))
axes[0].set_xlabel('Annual precipitation (mm)')
axes[0].set_ylabel('Accidents per 100k population')
axes[0].set_title('Precipitation vs accident rate')
axes[0].legend(title='Year', fontsize=9)
r0 = gy_wx['weather_annual_precip_mm'].corr(gy_wx['acc_per_100k'])
axes[0].annotate(f'r = {r0:.3f}', xy=(0.05, 0.93), xycoords='axes fraction',
                  fontsize=10, color=DK, fontweight='bold')

for yr, grp in gy_wx.groupby('year'):
    axes[1].scatter(grp['weather_mean_temp_mean_c'], grp['acc_per_100k'],
                    alpha=0.7, s=55, label=str(yr))
axes[1].set_xlabel('Mean annual temperature (deg C)')
axes[1].set_ylabel('Accidents per 100k population')
axes[1].set_title('Temperature vs accident rate')
axes[1].legend(title='Year', fontsize=9)
r1 = gy_wx['weather_mean_temp_mean_c'].corr(gy_wx['acc_per_100k'])
axes[1].annotate(f'r = {r1:.3f}', xy=(0.05, 0.93), xycoords='axes fraction',
                  fontsize=10, color=DK, fontweight='bold')

plt.tight_layout()
plt.show()
print(f'n = {len(gy_wx)} municipality-year observations with matched weather station')

---

## Infrastructure: how much cycling network is there, and does it help?

The OSM data gives us three types of cycling infrastructure per municipality: dedicated cycleways (`highway=cycleway`), separated tracks, and painted on-road lanes. The question is whether more infrastructure correlates with lower accident rates -- or whether more infrastructure just reflects more cycling activity.

In [ ]:
infra_prov = (
    infra.merge(gm[['gemeente_code', 'province']], on='gemeente_code', how='left')
    .groupby('province')[['km_cycleway', 'km_track', 'km_lane', 'bicycle_parking_count']]
    .sum().sort_values('km_cycleway', ascending=True)
)

fig, axes = plt.subplots(1, 2, figsize=(17, 6))

infra_prov[['km_cycleway', 'km_track', 'km_lane']].plot(
    kind='barh', stacked=True, ax=axes[0],
    color=[PR, SE, E1], edgecolor='white')
axes[0].set_title('Cycling infrastructure by province (km)')
axes[0].set_xlabel('Total km')
axes[0].legend(['Dedicated cycleway', 'Separated track', 'Painted lane'], fontsize=9)
axes[0].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))

bars = axes[1].barh(infra_prov.index, infra_prov['bicycle_parking_count'],
                     color=E2, edgecolor='white')
for bar, val in zip(bars, infra_prov['bicycle_parking_count']):
    axes[1].text(val + 5, bar.get_y() + bar.get_height() / 2,
                 f'{int(val):,}', va='center', fontsize=9, color=DK)
axes[1].set_title('Bicycle parking spots by province')
axes[1].set_xlabel('Parking spots')
axes[1].xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{int(x):,}'))

plt.tight_layout()
plt.show()
print(f'National total: {infra["km_total"].sum():,.0f} km of cycling infrastructure')

### Infrastructure density vs accident rate

In [ ]:
plot_df = gm_agg[
    (gm_agg['km_total'] > 0) &
    (gm_agg['accident_count'] > 0) &
    (gm_agg['population'] > 5_000)
].copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 5.5))

sc1 = axes[0].scatter(
    plot_df['infra_km_per_km2'], plot_df['acc_per_100k'],
    c=np.log1p(plot_df['population']), cmap='YlGn',
    alpha=0.7, s=40, edgecolors='none')
plt.colorbar(sc1, ax=axes[0], shrink=0.8, label='log(population)')
r1 = plot_df['infra_km_per_km2'].corr(plot_df['acc_per_100k'])
axes[0].annotate(f'r = {r1:.3f}', xy=(0.05, 0.92), xycoords='axes fraction',
                  fontsize=10, color=DK, fontweight='bold')
axes[0].set_xlabel('Infrastructure density (km / km2 area)')
axes[0].set_ylabel('Accidents per 100k population')
axes[0].set_title('More infrastructure -- safer or just more cycling?')

sc2 = axes[1].scatter(
    plot_df['km_total'], plot_df['fatal_per_100k'],
    c=plot_df['population_density_km2'], cmap='Purples',
    alpha=0.7, s=40, edgecolors='none')
plt.colorbar(sc2, ax=axes[1], shrink=0.8, label='Pop density (per km2)')
r2 = plot_df['km_total'].corr(plot_df['fatal_per_100k'])
axes[1].annotate(f'r = {r2:.3f}', xy=(0.05, 0.92), xycoords='axes fraction',
                  fontsize=10, color=DK, fontweight='bold')
axes[1].set_xlabel('Total cycling infrastructure (km)')
axes[1].set_ylabel('Fatal accidents per 100k')
axes[1].set_title('Total km vs fatality rate')

plt.tight_layout()
plt.show()

---

## Maps

Municipality boundaries are fetched live from PDOK (CBS Gebiedsindelingen, 2024 WFS). If that request fails, the maps fall back to centroid scatter plots.

In [ ]:
def fetch_gemeente_boundaries():
    url = (
        'https://service.pdok.nl/cbs/gebiedsindelingen/2024/wfs/v1_0'
        '?service=WFS&version=2.0.0&request=GetFeature'
        '&typeName=gebiedsindelingen:gemeente_gegeneraliseerd'
        '&outputFormat=application/json&srsName=EPSG:4326'
    )
    resp = requests.get(url, timeout=180)
    resp.raise_for_status()
    gdf = gpd.GeoDataFrame.from_features(resp.json().get('features', []), crs='EPSG:4326')
    gdf.columns = [c.lower() for c in gdf.columns]
    rename = {'statcode': 'gemeente_code', 'statnaam': 'gemeente_name',
              'gemeentecode': 'gemeente_code', 'gemeentenaam': 'gemeente_name'}
    gdf = gdf.rename(columns={k: v for k, v in rename.items() if k in gdf.columns})
    return gdf[['gemeente_code', 'gemeente_name', 'geometry']].copy()

try:
    boundaries = fetch_gemeente_boundaries()
    print(f'Boundaries: {len(boundaries)} gemeenten')
    HAS_BOUNDS = True
except Exception as exc:
    print(f'WFS unavailable ({exc}) -- using scatter fallback')
    HAS_BOUNDS = False

### Accident rate per 100k population by municipality

In [ ]:
if HAS_BOUNDS:
    m1 = boundaries.merge(gm_agg[['gemeente_code', 'acc_per_100k', 'accident_count']],
                           on='gemeente_code', how='left')
    fig, ax = plt.subplots(figsize=(11, 13))
    m1.plot(column='acc_per_100k', ax=ax, cmap='YlOrRd', legend=True,
             legend_kwds={'label': 'Accidents per 100k (2022-2024)', 'shrink': 0.55, 'format': '%.0f'},
             missing_kwds={'color': '#DDDDDD', 'label': 'No data'},
             edgecolor='white', linewidth=0.15)
    ax.set_title('Cycling accident rate per 100k population', fontsize=14, color=DK)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    top10 = gm_agg.nlargest(10, 'acc_per_100k')[['gemeente_name', 'province', 'acc_per_100k', 'accident_count']]
    print('Top 10 by accident rate:')
    print(top10.to_string(index=False))
else:
    sdf = gm_agg.merge(gm[['gemeente_code', 'centroid_lat', 'centroid_lon']], on='gemeente_code', how='left')
    fig, ax = plt.subplots(figsize=(10, 12))
    sc = ax.scatter(sdf['centroid_lon'], sdf['centroid_lat'], c=sdf['acc_per_100k'],
                     cmap='YlOrRd', s=30, alpha=0.8)
    plt.colorbar(sc, ax=ax, shrink=0.6, label='Accidents per 100k')
    ax.set_title('Accident rate per 100k (scatter -- boundaries unavailable)', color=DK)
    plt.tight_layout()
    plt.show()

### Fatal accidents -- rate vs absolute count

In [ ]:
if HAS_BOUNDS:
    m2 = boundaries.merge(gm_agg[['gemeente_code', 'fatal_per_100k', 'fatal_count']],
                           on='gemeente_code', how='left')
    m2['log_fatal'] = np.log1p(m2['fatal_count'].fillna(0))

    fig, axes = plt.subplots(1, 2, figsize=(18, 13))
    m2.plot(column='fatal_per_100k', ax=axes[0], cmap='Reds', legend=True,
             legend_kwds={'label': 'Fatal per 100k', 'shrink': 0.55, 'format': '%.3f'},
             missing_kwds={'color': '#DDDDDD'}, edgecolor='white', linewidth=0.15)
    axes[0].set_title('Fatal rate per 100k population', fontsize=13, color=DK)
    axes[0].axis('off')

    m2.plot(column='log_fatal', ax=axes[1], cmap='OrRd', legend=True,
             legend_kwds={'label': 'Fatal count (log)', 'shrink': 0.55,
                           'format': mtick.FuncFormatter(lambda x, _: f'{int(np.expm1(x))}')},
             missing_kwds={'color': '#DDDDDD'}, edgecolor='white', linewidth=0.15)
    axes[1].set_title('Absolute fatal accidents (log scale)', fontsize=13, color=DK)
    axes[1].axis('off')

    plt.tight_layout()
    plt.show()
else:
    sdf = gm_agg.merge(gm[['gemeente_code', 'centroid_lat', 'centroid_lon']], on='gemeente_code', how='left')
    fig, ax = plt.subplots(figsize=(10, 12))
    sc = ax.scatter(sdf['centroid_lon'], sdf['centroid_lat'], c=sdf['fatal_per_100k'],
                     cmap='Reds', s=40, alpha=0.8)
    plt.colorbar(sc, ax=ax, shrink=0.6, label='Fatal per 100k')
    ax.set_title('Fatal accident rate per 100k', color=DK)
    plt.tight_layout()
    plt.show()

### Cycling infrastructure density

In [ ]:
if HAS_BOUNDS:
    m3 = boundaries.merge(gm_agg[['gemeente_code', 'infra_km_per_km2']].fillna(0),
                           on='gemeente_code', how='left')
    fig, ax = plt.subplots(figsize=(11, 13))
    m3.plot(column='infra_km_per_km2', ax=ax, cmap='YlGn', legend=True,
             legend_kwds={'label': 'Infrastructure km per km2', 'shrink': 0.55, 'format': '%.2f'},
             missing_kwds={'color': '#DDDDDD'}, edgecolor='white', linewidth=0.15)
    ax.set_title('Cycling infrastructure density (km of routes per km2)', fontsize=14, color=DK)
    ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    sdf = gm_agg.merge(gm[['gemeente_code', 'centroid_lat', 'centroid_lon']], on='gemeente_code', how='left')
    fig, ax = plt.subplots(figsize=(10, 12))
    sc = ax.scatter(sdf['centroid_lon'], sdf['centroid_lat'], c=sdf['infra_km_per_km2'],
                     cmap='YlGn', s=30, alpha=0.8)
    plt.colorbar(sc, ax=ax, shrink=0.6, label='km / km2')
    ax.set_title('Infrastructure density', color=DK)
    plt.tight_layout()
    plt.show()

---

## Municipal risk index

A composite score built from four normalised indicators. Each is min-max scaled to [0, 1] before weighting:

- **40%** -- accidents per 100k population (higher = riskier)
- **30%** -- fatal accidents per 100k (higher = riskier)
- **20%** -- cycling infrastructure density, inverted (less infra = riskier)
- **10%** -- bicycle parking per km, inverted (less parking = riskier)

Final score is multiplied by 100 so it reads as 0-100. Only municipalities with at least one recorded accident are included.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

risk_df = gm_agg[gm_agg['accident_count'] > 0].copy()
risk_df['park_per_km'] = (
    risk_df['bicycle_parking_count'] / risk_df['km_total'].replace(0, np.nan)
).fillna(0)

sc = MinMaxScaler()
risk_df['s_acc']   = sc.fit_transform(risk_df[['acc_per_100k']])
risk_df['s_fatal'] = sc.fit_transform(risk_df[['fatal_per_100k']])
risk_df['s_infra'] = 1 - sc.fit_transform(risk_df[['infra_km_per_km2']])
risk_df['s_park']  = 1 - sc.fit_transform(risk_df[['park_per_km']])
risk_df['risk_score'] = (
    0.40 * risk_df['s_acc'] + 0.30 * risk_df['s_fatal'] +
    0.20 * risk_df['s_infra'] + 0.10 * risk_df['s_park']
) * 100
risk_df['risk_score'] = risk_df['risk_score'].round(1)
print(f'Municipalities scored: {len(risk_df)}')
print(f'Score range: {risk_df["risk_score"].min():.1f} -- {risk_df["risk_score"].max():.1f}')

In [ ]:
COLS = ['gemeente_name', 'province', 'risk_score', 'acc_per_100k', 'fatal_per_100k',
        'infra_km_per_km2', 'population']

top20  = risk_df.nlargest(20, 'risk_score')[COLS].reset_index(drop=True)
safe20 = risk_df.nsmallest(20, 'risk_score')[COLS].reset_index(drop=True)

print('=== 20 HIGHEST RISK ===')
print(top20.to_string(index=False))
print('\n=== 20 LOWEST RISK ===')
print(safe20.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 7))

r20 = risk_df.nlargest(20, 'risk_score').sort_values('risk_score')
bars = axes[0].barh(r20['gemeente_name'], r20['risk_score'], color=PR, edgecolor='white')
for bar, val in zip(bars, r20['risk_score']):
    axes[0].text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                 f'{val:.1f}', va='center', fontsize=8, color=DK)
axes[0].set_title('20 highest-risk municipalities')
axes[0].set_xlabel('Risk score (0-100)')
axes[0].set_xlim(0, 108)

s20 = risk_df.nsmallest(20, 'risk_score').sort_values('risk_score', ascending=False)
bars2 = axes[1].barh(s20['gemeente_name'], s20['risk_score'], color=SE, edgecolor='white')
for bar, val in zip(bars2, s20['risk_score']):
    axes[1].text(val + 0.3, bar.get_y() + bar.get_height() / 2,
                 f'{val:.1f}', va='center', fontsize=8, color=DK)
axes[1].set_title('20 lowest-risk municipalities')
axes[1].set_xlabel('Risk score (0-100)')
axes[1].set_xlim(0, 108)

plt.tight_layout()
plt.show()

### Correlation between risk factors

In [ ]:
corr_cols = {
    'acc_per_100k': 'Acc / 100k',
    'fatal_per_100k': 'Fatal / 100k',
    'infra_km_per_km2': 'Infra density',
    'park_per_km': 'Parking / km',
    'population_density_km2': 'Pop density',
    'risk_score': 'Risk score',
}
corr_df  = risk_df[list(corr_cols.keys())].rename(columns=corr_cols)
corr_mat = corr_df.corr()

cmap_c = mcolors.LinearSegmentedColormap.from_list(
    'corr', ['#2A4D40', '#F5F5F5', '#3D6E5C'], N=256)
fig, ax = plt.subplots(figsize=(8, 6.5))
mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
sns.heatmap(corr_mat, ax=ax, mask=mask, annot=True, fmt='.2f',
             annot_kws={'size': 10}, cmap=cmap_c, center=0,
             linewidths=0.5, linecolor='white', vmin=-1, vmax=1, square=True)
ax.set_title('Pairwise correlations -- risk factors')
plt.tight_layout()
plt.show()

---

## Code you can reuse

These are the building blocks for extensions of this analysis. Each function is short, documented, and self-contained.

### Filtering accidents

In [ ]:
def filter_accidents(df, province=None, year=None, severity=None, weather=None):
    """Return a filtered slice of the accidents DataFrame."""
    mask = pd.Series(True, index=df.index)
    if province: mask &= df['province'].isin([province] if isinstance(province, str) else province)
    if year:     mask &= df['year'].isin([year] if isinstance(year, int) else year)
    if severity: mask &= df['severity_label'].isin([severity] if isinstance(severity, str) else severity)
    if weather:  mask &= df['weather_label'].isin([weather] if isinstance(weather, str) else weather)
    return df[mask].copy()

# example
sample = filter_accidents(acc, province='Noord-Holland', year=2023, severity='fatal')
print(f'Fatal 2023, Noord-Holland: {len(sample)} accidents')
print(sample[['gemeente_name_raw', 'road_situation', 'speed_limit_kmh']].head())

### Fatal rate grouped by any column

In [ ]:
def severity_rate(df, group_col, severity='fatal'):
    """Return severity rate (%) grouped by any column."""
    grouped = df.groupby(group_col)
    rate = (
        grouped['severity_label']
        .apply(lambda s: (s == severity).sum() / len(s) * 100)
        .rename(f'{severity}_pct').reset_index()
    )
    rate['n'] = grouped.size().values
    return rate.sort_values(f'{severity}_pct', ascending=False)

print(severity_rate(acc, 'road_en').to_string(index=False))

### Join accidents to municipality attributes

In [ ]:
acc_enriched = (
    acc
    .merge(gm[['gemeente_code', 'population', 'population_density_km2', 'area_km2']],
           on='gemeente_code', how='left')
    .merge(infra[['gemeente_code', 'km_total', 'bicycle_parking_count']],
           on='gemeente_code', how='left')
)
print(f'Enriched: {len(acc_enriched):,} rows  x  {acc_enriched.shape[1]} cols')
print('Added columns:', [c for c in acc_enriched.columns if c not in acc.columns])

### Build a choropleth from any municipality-level metric

In [ ]:
def choropleth(gdf, value_col, title, cmap='YlOrRd', label=None, log_scale=False):
    """Plot a municipality choropleth. gdf must have gemeente_code + geometry."""
    plot_col = value_col
    if log_scale:
        gdf = gdf.copy()
        gdf['_log'] = np.log1p(gdf[value_col].fillna(0))
        plot_col = '_log'
    fig, ax = plt.subplots(figsize=(10, 12))
    gdf.plot(column=plot_col, ax=ax, cmap=cmap, legend=True,
              legend_kwds={'label': label or value_col, 'shrink': 0.55},
              missing_kwds={'color': '#DDDDDD'}, edgecolor='white', linewidth=0.15)
    ax.set_title(title, fontsize=14, color=DK)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

# Uncomment if boundaries are available:
# map_df = boundaries.merge(gm_agg[['gemeente_code', 'acc_per_100k']], on='gemeente_code', how='left')
# choropleth(map_df, 'acc_per_100k', 'Accident rate per 100k', label='Accidents / 100k')
print('choropleth() ready')

---

## Data sources and notes

| Source | What was used | Licence |
|--------|---------------|---------|
| RWS BRON | Cycling accident microdata, WFS 2022-2024 | CC BY 4.0 |
| KNMI | Daily station observations, daggegevens API | CC BY 4.0 |
| CBS / PDOK | Municipality boundaries (Gebiedsindelingen 2024 WFS) | CC BY 4.0 |
| CBS | Kerncijfers wijken en buurten 2024 (85984NED) | CC BY 4.0 |
| OpenStreetMap | Cycling infrastructure via Overpass API | ODbL 1.0 |

**Known limitations**

- BRON covers only police-reported accidents on public roads. Single-vehicle falls without police contact are not captured.
- Weather is matched by nearest KNMI station. Only 38 stations cover 342 municipalities, so most municipalities share a station and the weather columns in `gemeente_yearly_summary` are sparse (~10% of rows have non-null values).
- OSM infrastructure coverage is excellent nationally but varies by municipality. Rural areas may be under-mapped.
- Population figures are CBS 2024 registered residents, not cycling exposure.